# MedGemma-27B Response Collection
## Thesis Phase II — standalone collection notebook

**Before running:**
1. Accept the gated licence at https://huggingface.co/google/medgemma-27b-it
2. Set Runtime → Change runtime type → **T4 GPU** (required!)
3. Paste your HF_TOKEN in the config cell below

Output: `medgemma_responses.json` in your evaluation folder.

In [ ]:
!pip install -q transformers accelerate bitsandbytes tqdm
print("✅ Done")

✅ Done


In [ ]:
import os, json, time
import torch
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
print(f"CUDA: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — switch to T4 GPU!'}")

CUDA: True — NVIDIA H100 80GB HBM3


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE     = Path("/content/drive/MyDrive/medical_protocols")
OUT_DIR  = BASE / "evaluation"
OUT_DIR.mkdir(parents=True, exist_ok=True)
QUESTIONS_JSON = BASE / "question_design_v1" / "final_questions.json"
print(f"OUT_DIR: {OUT_DIR}")

OUT_DIR: /content/drive/MyDrive/medical_protocols/evaluation


In [ ]:

HF_TOKEN = ""   # ← paste here, e.g. "hf_..."

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass

print("HF_TOKEN:", "✅ set" if HF_TOKEN else "❌ MISSING — paste above")

HF_TOKEN: ✅ set


In [ ]:
SYSTEM_PROMPT = (
    "You are a clinical assistant specialising in obstetrics and gynaecology. "
    "Answer the following question based on evidence-based clinical guidelines. "
    "Be specific and include: diagnostic criteria, medication names and dosages "
    "where relevant, gestational age thresholds, contraindications, and "
    "recommended follow-up intervals. "
    "Do not give general health advice — provide protocol-level clinical detail."
)

print("System prompt:")
print("-" * 60)
print(SYSTEM_PROMPT)
print("-" * 60)
print(f"Length: {len(SYSTEM_PROMPT)} characters")

System prompt:
------------------------------------------------------------
You are a clinical assistant specialising in obstetrics and gynaecology. Answer the following question based on evidence-based clinical guidelines. Be specific and include: diagnostic criteria, medication names and dosages where relevant, gestational age thresholds, contraindications, and recommended follow-up intervals. Do not give general health advice — provide protocol-level clinical detail.
------------------------------------------------------------
Length: 398 characters


In [ ]:
questions = json.loads(QUESTIONS_JSON.read_text(encoding="utf-8"))
print(f"Loaded {len(questions)} questions")

Loaded 15 questions


In [ ]:
# ── Load MedGemma-27B in 4-bit NF4 quantization ─────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("google/medgemma-27b-it", token=HF_TOKEN)
print("Loading model (4-bit NF4) — this takes ~10 min on first run...")
model = AutoModelForCausalLM.from_pretrained(
    "google/medgemma-27b-it",
    quantization_config=bnb_cfg,
    device_map="auto",
    token=HF_TOKEN,
)
print("✅ MedGemma-27B loaded!")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

Loading tokenizer...
Loading model (4-bit NF4) — this takes ~10 min on first run...


Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

✅ MedGemma-27B loaded!
GPU memory used: 32.6 GB


In [ ]:
def query_medgemma(question: str, system_prompt: str) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": question},
    ]
    encoded_inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **encoded_inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][encoded_inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("Testing MedGemma with a short query...")
test = query_medgemma("What is preeclampsia?", SYSTEM_PROMPT)
print(f"Response ({len(test)} chars): {test[:200]}...")

Testing MedGemma with a short query...
Response (2036 chars): Okay, I can provide a detailed clinical overview of preeclampsia based on evidence-based guidelines.

**Preeclampsia: A Clinical Definition and Management Protocol**

**Definition:**

Preeclampsia is ...


In [ ]:
SAVE_PATH = OUT_DIR / "medgemma_responses.json"

if SAVE_PATH.exists():
    collected = json.loads(SAVE_PATH.read_text(encoding="utf-8"))
    print(f"Resumed: {len(collected)} existing responses found.")
else:
    collected = []
    print("Starting fresh.")

done_keys = {(r["question_id"], r["language"]) for r in collected}
pending = [(q, lang) for q in questions for lang in ["EN", "RU"]
           if (q["id"], lang) not in done_keys]
print(f"Pending: {len(pending)} / 30 queries")

for q, lang in tqdm(pending, desc="MedGemma-27B"):
    q_text = q["question_en"] if lang == "EN" else q["question_ru"]
    try:
        response_text = query_medgemma(q_text, SYSTEM_PROMPT)
    except Exception as e:
        response_text = f"ERROR: {e}"
        print(f"  ❌ {q['id']} {lang}: {e}")

    collected.append({
        "bot":             "MedGemma-27B",
        "question_id":     q["id"],
        "stratum":         q.get("stratum", ""),
        "language":        lang,
        "question_text":   q_text,
        "response_text":   response_text,
        "target_protocol": q["target_protocol"],
        "timestamp":       datetime.utcnow().isoformat(),
    })
    SAVE_PATH.write_text(json.dumps(collected, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\n✅ Done! {len(collected)}/30 saved to {SAVE_PATH}")
errors = [r for r in collected if r["response_text"].startswith("ERROR:")]
print(f"Errors: {len(errors)}")
if not errors: print("All clean ✅")

Resumed: 30 existing responses found.
Pending: 0 / 30 queries


MedGemma-27B: 0it [00:00, ?it/s]


✅ Done! 30/30 saved to /content/drive/MyDrive/medical_protocols/evaluation/medgemma_responses.json
Errors: 0
All clean ✅


In [ ]:
# ── Ad-hoc cell: run updated case-based questions through MedGemma ─────────

UPDATED_QUESTIONS = [
    {
        "question_id": "A1", "stratum": "Diagnosis & Criteria",
        "language": "EN",
        "question_text": "A pregnant woman at 34 weeks presents with blood pressure 168/112 mmHg and proteinuria. According to clinical guidelines, does this meet the criteria for severe preeclampsia and what additional features confirm the diagnosis?",
        "target_protocol": "en_Гипертензивные_состояния_при_беременности.md"
    },
    {
        "question_id": "A1", "stratum": "Diagnosis & Criteria",
        "language": "RU",
        "question_text": "Беременная на сроке 34 недели имеет артериальное давление 168/112 мм рт. ст. и протеинурию. Соответствует ли это критериям тяжёлой преэклампсии согласно клиническим протоколам и какие дополнительные признаки подтверждают диагноз?",
        "target_protocol": "en_Гипертензивные_состояния_при_беременности.md"
    },
    {
        "question_id": "A2", "stratum": "Diagnosis & Criteria",
        "language": "EN",
        "question_text": "A pregnant patient undergoes a 75-g oral glucose tolerance test at 26 weeks. The results are: fasting glucose 5.2 mmol/L, 1-hour 10.1 mmol/L, and 2-hour 8.7 mmol/L. According to clinical guidelines, do these values meet the diagnostic criteria for gestational diabetes mellitus?",
        "target_protocol": "en_Сахарный_диабет_при_беременности,_в_родах_и_послеродовом_периоде.md"
    },
    {
        "question_id": "A2", "stratum": "Diagnosis & Criteria",
        "language": "RU",
        "question_text": "Беременная проходит пероральный глюкозотолерантный тест (75 г) на сроке 26 недель. Результаты: глюкоза натощак 5,2 ммоль/л, через 1 час 10,1 ммоль/л, через 2 часа 8,7 ммоль/л. Соответствуют ли эти показатели диагностическим критериям гестационного сахарного диабета согласно клиническим протоколам?",
        "target_protocol": "en_Сахарный_диабет_при_беременности,_в_родах_и_послеродовом_периоде.md"
    },
    {
        "question_id": "D1", "stratum": "Monitoring & Screening",
        "language": "EN",
        "question_text": "A pregnant woman attends her first antenatal visit at 10 weeks of gestation. According to clinical guidelines, what screening tests should be performed during the first trimester and at what gestational weeks are they recommended?",
        "target_protocol": "en_Антенатальный_уход.md"
    },
    {
        "question_id": "D1", "stratum": "Monitoring & Screening",
        "language": "RU",
        "question_text": "Беременная женщина впервые обращается за дородовым наблюдением на сроке 10 недель. Согласно клиническим протоколам, какие скрининговые исследования должны быть проведены в первом триместре и на каких сроках беременности они рекомендуются?",
        "target_protocol": "en_Антенатальный_уход.md"
    },
    {
        "question_id": "D2", "stratum": "Monitoring & Screening",
        "language": "EN",
        "question_text": "A pregnant woman with gestational diabetes measures fasting glucose levels of 5.6 mmol/L during home monitoring. According to clinical guidelines, is this within the recommended glycaemic targets and what management adjustments may be required?",
        "target_protocol": "en_Сахарный_диабет_при_беременности,_в_родах_и_послеродовом_периоде.md"
    },
    {
        "question_id": "D2", "stratum": "Monitoring & Screening",
        "language": "RU",
        "question_text": "Беременная с гестационным сахарным диабетом измеряет уровень глюкозы натощак 5,6 ммоль/л при домашнем мониторинге. Соответствует ли это рекомендуемым целевым показателям гликемии согласно клиническим протоколам и какие коррективы в лечении могут потребоваться?",
        "target_protocol": "en_Сахарный_диабет_при_беременности,_в_родах_и_послеродовом_периоде.md"
    },
]

SAVE_PATH = OUT_DIR / "medgemma_responses.json"
existing = json.loads(SAVE_PATH.read_text(encoding="utf-8")) if SAVE_PATH.exists() else []

existing_keys = {(r["question_id"], r["language"]): i for i, r in enumerate(existing)}

from datetime import datetime
from tqdm.auto import tqdm

print(f"Running {len(UPDATED_QUESTIONS)} updated questions through MedGemma-27B...\n")

for q in tqdm(UPDATED_QUESTIONS, desc="MedGemma updated Qs"):
    qid  = q["question_id"]
    lang = q["language"]
    print(f"  {qid} {lang} ...", end=" ", flush=True)
    response_text = query_medgemma(q["question_text"], SYSTEM_PROMPT)
    record = {
        "bot":           "MedGemma-27B",
        "question_id":   qid,
        "stratum":       q["stratum"],
        "language":      lang,
        "question_text": q["question_text"],
        "response_text": response_text,
        "target_protocol": q["target_protocol"],
        "timestamp":     datetime.utcnow().isoformat(),
    }
    key = (qid, lang)
    if key in existing_keys:
        existing[existing_keys[key]] = record  # overwrite old answer
        print(f"updated ({len(response_text)} chars)")
    else:
        existing.append(record)
        existing_keys[key] = len(existing) - 1
        print(f"added ({len(response_text)} chars)")
    SAVE_PATH.write_text(json.dumps(existing, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\n✅ Done. medgemma_responses.json now has {len(existing)} records.")
print("Preview of updated responses:")
for q in UPDATED_QUESTIONS:
    rec = next(r for r in existing if r["question_id"]==q["question_id"] and r["language"]==q["language"])
    print(f"  {rec['question_id']} {rec['language']}: {rec['response_text'][:120]}...")


Running 8 updated questions through MedGemma-27B...



MedGemma updated Qs:   0%|          | 0/8 [00:00<?, ?it/s]

  A1 EN ... updated (2100 chars)
  A1 RU ... 

/tmp/ipykernel_753/1243817512.py:81: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":     datetime.utcnow().isoformat(),


updated (2069 chars)
  A2 EN ... updated (1597 chars)
  A2 RU ... updated (1571 chars)
  D1 EN ... updated (2146 chars)
  D1 RU ... updated (2222 chars)
  D2 EN ... updated (2058 chars)
  D2 RU ... updated (1436 chars)

✅ Done. medgemma_responses.json now has 30 records.
Preview of updated responses:
  A1 EN: Okay, let's break down the diagnosis of severe preeclampsia in this specific clinical scenario based on evidence-based g...
  A1 RU: Okay, let's break down the clinical scenario and determine if the patient meets the criteria for severe preeclampsia acc...
  A2 EN: Okay, let's analyze the patient's 75-g oral glucose tolerance test (OGTT) results at 26 weeks gestation according to cur...
  A2 RU: Okay, let's analyze the results of the 75g Oral Glucose Tolerance Test (OGTT) performed at 26 weeks gestation according ...
  D1 EN: Okay, as a clinical assistant specialising in obstetrics and gynaecology, I can provide a detailed protocol for first-tr...
  D1 RU: Okay, let's outline the 

In [ ]:
# ── Ad-hoc cell: re-run 6 RU refusals from MedGemma ────────────────────────


SYSTEM_PROMPT_RU_SOFT = (
    "Вы — ассистент клинициста, специализирующийся в акушерстве и гинекологии. "
    "Вы помогаете в академических и образовательных целях: анализируете клинические протоколы "
    "и отвечаете на вопросы на основе доказательной медицины. "
    "Отвечайте на русском языке, давайте конкретные клинические детали: "
    "диагностические критерии, названия препаратов и дозировки, пороговые значения "
    "гестационного возраста, противопоказания и интервалы наблюдения. "
    "Не давайте общих советов — предоставляйте клинические детали уровня протокола."
)

REFUSAL_QUESTIONS = [
    {
        "question_id": "A3", "stratum": "Diagnosis & Criteria",
        "language": "RU",
        "question_text": "Какие клинические и лабораторные критерии определяют септический шок у акушерских пациенток и какие шкалы оценки рекомендованы?",
        "target_protocol": "en_Акушерский_сепсис.md"
    },
    {
        "question_id": "B1", "stratum": "Pharmacotherapy",
        "language": "RU",
        "question_text": "Какова рекомендуемая антигипертензивная терапия первой линии при тяжёлой преэклампсии и каковы конкретные дозировки?",
        "target_protocol": "en_Гипертензивные_состояния_при_беременности.md"
    },
    {
        "question_id": "B2", "stratum": "Pharmacotherapy",
        "language": "RU",
        "question_text": "Какая эмпирическая антибиотикотерапия рекомендована при акушерском сепсисе и в какие сроки она должна быть начата?",
        "target_protocol": "en_Акушерский_сепсис.md"
    },
    {
        "question_id": "C1", "stratum": "Surgical & Emergency",
        "language": "RU",
        "question_text": "Каковы поэтапные хирургические вмешательства, рекомендованные при послеродовом кровотечении, не отвечающем на утеротоническую терапию?",
        "target_protocol": "en_Послеродовое_кровотечение.md"
    },
    {
        "question_id": "C2", "stratum": "Surgical & Emergency",
        "language": "RU",
        "question_text": "Каковы показания к экстренной гистерэктомии у акушерских пациенток согласно клиническим протоколам?",
        "target_protocol": "en_Послеродовое_кровотечение.md"
    },
    {
        "question_id": "D3", "stratum": "Monitoring & Screening",
        "language": "RU",
        "question_text": "Какие факторы риска позволяют выявить беременных женщин, которым показана профилактика прогестероном для предотвращения преждевременных родов?",
        "target_protocol": "en_Преждевременные_роды.md"
    },
]

SAVE_PATH = OUT_DIR / "medgemma_responses.json"
existing = json.loads(SAVE_PATH.read_text(encoding="utf-8")) if SAVE_PATH.exists() else []
existing_keys = {(r["question_id"], r["language"]): i for i, r in enumerate(existing)}

print(f"Re-running {len(REFUSAL_QUESTIONS)} refused RU questions with soft prompt...\n")

from tqdm.auto import tqdm
from datetime import datetime

for q in tqdm(REFUSAL_QUESTIONS, desc="MedGemma RU fix"):
    qid  = q["question_id"]
    lang = q["language"]
    print(f"  {qid} {lang} ...", end=" ", flush=True)
    response_text = query_medgemma(q["question_text"], SYSTEM_PROMPT_RU_SOFT)

    # Check if it's still a refusal
    refusal_phrases = ["не могу", "языковая модель", "проконсультируйтесь", "медицинские советы"]
    is_refusal = any(p in response_text.lower() for p in refusal_phrases)
    if is_refusal:
        print(f"⚠️  still refusing ({len(response_text)} chars) — keeping anyway")
    else:
        print(f"✅ answered ({len(response_text)} chars)")

    record = {
        "bot":             "MedGemma-27B",
        "question_id":     qid,
        "stratum":         q["stratum"],
        "language":        lang,
        "question_text":   q["question_text"],
        "response_text":   response_text,
        "target_protocol": q["target_protocol"],
        "timestamp":       datetime.utcnow().isoformat(),
    }
    key = (qid, lang)
    if key in existing_keys:
        existing[existing_keys[key]] = record
    else:
        existing.append(record)
        existing_keys[key] = len(existing) - 1
    SAVE_PATH.write_text(json.dumps(existing, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"\n✅ Done. medgemma_responses.json updated.")
print("\nFinal check on fixed responses:")
for q in REFUSAL_QUESTIONS:
    rec = next(r for r in existing if r["question_id"]==q["question_id"] and r["language"]==q["language"])
    preview = rec["response_text"][:120].replace("\n", " ")
    print(f"  {rec['question_id']} {rec['language']}: {preview}...")


Re-running 6 refused RU questions with soft prompt...



MedGemma RU fix:   0%|          | 0/6 [00:00<?, ?it/s]

  A3 RU ... ✅ answered (1590 chars)
  B1 RU ... 

/tmp/ipykernel_753/4068604901.py:86: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":       datetime.utcnow().isoformat(),


✅ answered (1669 chars)
  B2 RU ... ✅ answered (1870 chars)
  C1 RU ... ✅ answered (1709 chars)
  C2 RU ... ⚠️  still refusing (1755 chars) — keeping anyway
  D3 RU ... ✅ answered (1706 chars)

✅ Done. medgemma_responses.json updated.

Final check on fixed responses:
  A3 RU: Здравствуйте! Я готов предоставить информацию о клинических и лабораторных критериях септического шока у акушерских паци...
  B1 RU: Здравствуйте! Я готов предоставить информацию об антигипертензивной терапии первой линии при тяжёлой преэклампсии, основ...
  B2 RU: Здравствуйте! Я готов предоставить информацию об эмпирической антибиотикотерапии при акушерском сепсисе, основанную на с...
  C1 RU: Хорошо, давайте разберем поэтапные хирургические вмешательства при послеродовом кровотечении (ППК), которое не поддается...
  C2 RU: Хорошо, давайте рассмотрим показания к экстренной гистерэктомии у акушерских пациенток, основываясь на клинических прото...
  D3 RU: Здравствуйте! Я готов предоставить информацию о факторах р